In [1]:
# 目的: 让语言模型决定使用哪个函数以及确定函数的输入参数


import os

from langchain_community.utilities.openapi import OpenAPISpec

api_key = os.environ.get("DEEPSEEK_API_KEY")
model = "deepseek-v4-flash"
base_url = "https://api.deepseek.com/"

C:\Users\hilary\AppData\Local\Temp\ipykernel_18200\708220698.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities.openapi import OpenAPISpec


In [2]:
# 导入工具装饰器
from langchain_classic.agents import tool

In [3]:
@tool
def search(query: str) -> str:
    """Search for weather online"""
    return "42f"

In [4]:
search.name

'search'

In [5]:
search.description

'Search for weather online'

In [6]:
search.args

{'query': {'title': 'Query', 'type': 'string'}}

In [7]:
# 明确输入模式
from pydantic import BaseModel, Field
class SearchInput(BaseModel):
    query: str = Field(description="Thing to Search for")

In [8]:
@tool(args_schema=SearchInput)
def search(query: str) -> str:
    """Search for the weather online."""
    return "42f"

In [9]:
search.args

{'query': {'description': 'Thing to Search for',
  'title': 'Query',
  'type': 'string'}}

In [10]:
search.run("sf")

'42f'

In [11]:
# 创建一个实时获取天气的工具
import requests
import datetime

# define the input schema
class OpenMeteoInput(BaseModel):
    latitude: float = Field(..., description="Latitude of the location to fetch weather data for")
    longitude: float = Field(..., description="Longitude of the location to fetch weather data for")

@tool(args_schema=OpenMeteoInput)
def get_current_temperature(latitude: float, longitude: float) -> dict:
    """Fetch current temperature for given coordinates."""

    BASE_URL = "https://api.open-meteo.com/v1/forecast"

    params = {
        'latitude': latitude,
        'longitude': longitude,
        'hourly': 'temperature_2m',
        'forecast_days': 1,
    }

    # 创建请求
    response = requests.get(BASE_URL, params=params)

    if response.status_code == 200:
        results = response.json()
    else :
        raise Exception(f"API Request failed with status code: {response.status_code}")


    current_utc_time = datetime.datetime.now(datetime.timezone.utc)
    time_list = [datetime.datetime.fromisoformat(t.replace("Z", "+00:00")).replace(tzinfo=datetime.timezone.utc) for t in results["hourly"]["time"]]
    temperature_list = results['hourly']['temperature_2m']

    closeset_time_index = min(range(len(time_list)), key=lambda i: abs(time_list[i] - current_utc_time))

    current_temperature = temperature_list[closeset_time_index]

    return f"The current temperature is {current_temperature} ℃"

In [12]:
get_current_temperature.name

'get_current_temperature'

In [13]:
get_current_temperature.description

'Fetch current temperature for given coordinates.'

In [14]:
get_current_temperature.args

{'latitude': {'description': 'Latitude of the location to fetch weather data for',
  'title': 'Latitude',
  'type': 'number'},
 'longitude': {'description': 'Longitude of the location to fetch weather data for',
  'title': 'Longitude',
  'type': 'number'}}

In [15]:
from langchain_classic.tools import format_tool_to_openai_function

In [16]:
format_tool_to_openai_function(get_current_temperature)

{'name': 'get_current_temperature',
 'description': 'Fetch current temperature for given coordinates.',
 'parameters': {'properties': {'latitude': {'description': 'Latitude of the location to fetch weather data for',
    'type': 'number'},
   'longitude': {'description': 'Longitude of the location to fetch weather data for',
    'type': 'number'}},
  'required': ['latitude', 'longitude'],
  'type': 'object'}}

In [17]:
get_current_temperature.invoke({"latitude": 13, "longitude": 14})

'The current temperature is 33.7 ℃'

In [18]:
# 定义一个链接维基百科工具
import wikipedia
wikipedia.wikipedia.API_URL = "https://en.wikipedia.org/w/api.php"
@tool
def search_wikipedia(query: str) -> str:
    """Run Wikipedia search and get page summaries"""
    page_titles = wikipedia.search(query);
    summaries = []
    for page_title in page_titles[:3]: # 仅遍历前三个
        try:
            wiki_page = wikipedia.page(title=page_title, auto_suggest=False)
            summaries.append(f"Page:{page_title}\nSummary:{wiki_page.summary}")
        except(
            wikipedia.exceptions.PageError,
            wikipedia.exceptions.DisambiguationError,
        ):
            pass

    if not  summaries:
        return "No good Wikipedia Search Result was found"
    return "\n\n".join(summaries)

In [19]:
search_wikipedia.name

'search_wikipedia'

In [20]:
search_wikipedia.description

'Run Wikipedia search and get page summaries'

In [21]:
format_tool_to_openai_function(search_wikipedia)

{'name': 'search_wikipedia',
 'description': 'Run Wikipedia search and get page summaries',
 'parameters': {'properties': {'query': {'type': 'string'}},
  'required': ['query'],
  'type': 'object'}}

In [22]:
# 输入query调用wikipedia_search
# search_wikipedia.invoke({"query":"langchain"}) # 需要开启代理

In [23]:
# 通过函数API
from langchain_classic.chains.openai_functions.openapi import openapi_spec_to_openai_fn
from langchain_community.utilities.openapi import OpenAPISpec

In [24]:
text = """
{
  "openapi": "3.0.0",
  "info": {
    "version": "1.0.0",
    "title": "Swagger Petstore",
    "license": {
      "name": "MIT"
    }
  },
  "servers": [
    {
      "url": "http://petstore.swagger.io/v1"
    }
  ],
  "paths": {
    "/pets": {
      "get": {
        "summary": "List all pets",
        "operationId": "listPets",
        "tags": [
          "pets"
        ],
        "parameters": [
          {
            "name": "limit",
            "in": "query",
            "description": "How many items to return at one time (max 100)",
            "required": false,
            "schema": {
              "type": "integer",
              "maximum": 100,
              "format": "int32"
            }
          }
        ],
        "responses": {
          "200": {
            "description": "A paged array of pets",
            "headers": {
              "x-next": {
                "description": "A link to the next page of responses",
                "schema": {
                  "type": "string"
                }
              }
            },
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Pets"
                }
              }
            }
          },
          "default": {
            "description": "unexpected error",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Error"
                }
              }
            }
          }
        }
      },
      "post": {
        "summary": "Create a pet",
        "operationId": "createPets",
        "tags": [
          "pets"
        ],
        "responses": {
          "201": {
            "description": "Null response"
          },
          "default": {
            "description": "unexpected error",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Error"
                }
              }
            }
          }
        }
      }
    },
    "/pets/{petId}": {
      "get": {
        "summary": "Info for a specific pet",
        "operationId": "showPetById",
        "tags": [
          "pets"
        ],
        "parameters": [
          {
            "name": "petId",
            "in": "path",
            "required": true,
            "description": "The id of the pet to retrieve",
            "schema": {
              "type": "string"
            }
          }
        ],
        "responses": {
          "200": {
            "description": "Expected response to a valid request",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Pet"
                }
              }
            }
          },
          "default": {
            "description": "unexpected error",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Error"
                }
              }
            }
          }
        }
      }
    }
  },
  "components": {
    "schemas": {
      "Pet": {
        "type": "object",
        "required": [
          "id",
          "name"
        ],
        "properties": {
          "id": {
            "type": "integer",
            "format": "int64"
          },
          "name": {
            "type": "string"
          },
          "tag": {
            "type": "string"
          }
        }
      },
      "Pets": {
        "type": "array",
        "items": {
          "$ref": "#/components/schemas/Pet"
        }
      },
      "Error": {
        "type": "object",
        "required": [
          "code",
          "message"
        ],
        "properties": {
          "code": {
            "type": "integer",
            "format": "int32"
          },
          "message": {
            "type": "string"
          }
        }
      }
    }
  }
}
"""

In [25]:
spec = OpenAPISpec.from_text(text)

Attempting to load an OpenAPI 3.0.0 spec.  This may result in degraded performance. Convert your OpenAPI spec to 3.1.* spec for better support.


In [26]:
pet_openai_functions, pet_callables = openapi_spec_to_openai_fn(spec) # 获得函数定义，并得到函数调用

C:\Users\hilary\AppData\Local\Temp\ipykernel_18200\2401438846.py:1: LangChainDeprecationWarning: The function `openapi_spec_to_openai_fn` was deprecated in LangChain 1.0.4 and will be removed in 2.0.0 Bind your OpenAPI operations as tools on a chat model with `ChatModel.bind_tools(...)` and execute the resulting tool calls with an HTTP client (e.g. `requests` or `httpx`).
  pet_openai_functions, pet_callables = openapi_spec_to_openai_fn(spec) # 获得函数定义，并得到函数调用


In [27]:
pet_openai_functions # 获得了三个函数

[{'name': 'listPets',
  'description': 'List all pets',
  'parameters': {'type': 'object',
   'properties': {'params': {'type': 'object',
     'properties': {'limit': {'type': 'integer',
       'maximum': 100.0,
       'schema_format': 'int32',
       'description': 'How many items to return at one time (max 100)'}},
     'required': []}}}},
 {'name': 'createPets',
  'description': 'Create a pet',
  'parameters': {'type': 'object', 'properties': {}}},
 {'name': 'showPetById',
  'description': 'Info for a specific pet',
  'parameters': {'type': 'object',
   'properties': {'path_params': {'type': 'object',
     'properties': {'petId': {'type': 'string',
       'description': 'The id of the pet to retrieve'}},
     'required': ['petId']}}}}]

In [28]:
# 实验语言模型调用函数
from langchain_openai.chat_models import ChatOpenAI

In [29]:
# 创建语言模型
model = ChatOpenAI(
    base_url=base_url,
    api_key=api_key,
    model=model,
    temperature=0
    # extra_body={
    #     "thinking": {"thinking":"disabled"},
    # }
).bind_tools(pet_openai_functions)

In [30]:
response1 = model.invoke("what are three pets names")

In [31]:
response1.tool_calls[0]["args"]

{'params': {'limit': 100}}

In [32]:
response2 = model.invoke("tell me about pet with id 42")

In [33]:
response2

AIMessage(content="I'll look up that pet for you.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 444, 'total_tokens': 516, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 18, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 256, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 188}, 'model_provider': 'openai', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'b991a34e-c326-4ad6-9a01-aa3a6d9a3e84', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0c875-1462-70d1-a505-bad595a7fb5e-0', tool_calls=[{'name': 'showPetById', 'args': {'path_params': {'petId': '42'}}, 'id': 'call_00_6qaIy89HLtGV3cwihTl95217', 'type': 'tool_call'}], invalid_tool_calls=[

In [34]:
# 使用前文创建的天气函数和wikipedia函数
# 创建openai函数的规范列表
functions = [
    format_tool_to_openai_function(f) for f in [
        search_wikipedia, get_current_temperature
    ]
]

# 绑定工具，使用传统写法
model = ChatOpenAI(
    model="deepseek-v4-flash",
    base_url=base_url,
    api_key=api_key,
    temperature=0
).bind_tools(functions)

In [35]:
response = model.invoke("what's the weather in sf right now")

In [36]:
model.invoke("what's the weather in sf right now")

AIMessage(content="I'll check the current temperature in San Francisco for you.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 391, 'total_tokens': 493, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 27, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 256, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 135}, 'model_provider': 'openai', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '02b2621a-fa95-4b1d-b5ad-eb98e94ad91c', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0c875-1a06-7e42-9088-61e664f8476b-0', tool_calls=[{'name': 'get_current_temperature', 'args': {'latitude': 37.7749, 'longitude': -122.4194}, 'id': 'call_00_naeR6YnXIBwP1L

In [37]:
response

AIMessage(content="I'll check that for you.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 391, 'total_tokens': 484, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 23, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 256, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 135}, 'model_provider': 'openai', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '2c10b55e-d838-4e4c-a192-571239b4721a', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0c875-1735-7cb3-8b2f-458e92b0ddee-0', tool_calls=[{'name': 'get_current_temperature', 'args': {'latitude': 37.7749, 'longitude': -122.4194}, 'id': 'call_00_x0YK2uj3jHutxbGqqAlD5447', 'type': 'tool_call'}], i

In [38]:
model.invoke("what is langchain")

AIMessage(content='**LangChain** is an open-source framework for building applications powered by large language models (LLMs). It was created by Harrison Chase in late 2022 and has become one of the most popular tools in the LLM/AI application ecosystem.\n\n## What it does\n\nAt its core, LangChain provides standard building blocks and abstractions that make it easier to connect LLMs to other data sources and let them interact with the world. Instead of writing custom glue code for every project, you get reusable components.\n\n## Key components\n\n- **Models** — Standardized interfaces for chatting with LLMs (OpenAI, Anthropic, local models, etc.)\n- **Prompts** — Templates for structuring inputs to models\n- **Chains** — Sequences of calls (e.g., prompt → LLM → output parser) that can be composed together\n- **Memory** — Mechanisms for persisting state across interactions (e.g., conversation history)\n- **Retrieval (RAG)** — Tools for connecting LLMs to external data (documents, dat

In [39]:
# 在调用模型前添加prompt
from langchain_classic.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are helpful but essay assistant"),
        ("user","{input}"),
    ]
)

# 使用LCEL构造chain
chain = prompt | model # 将prompt传递给model

In [40]:
response = chain.invoke({"input":"what is the weather in sf right now?"})


In [41]:
response.tool_calls

[{'name': 'get_current_temperature',
  'args': {'latitude': 37.7749, 'longitude': -122.4194},
  'id': 'call_00_dgn3yvFtSIYkVa4c6Wsi5167',
  'type': 'tool_call'}]

In [42]:
# 解析器,解析模型输出
from langchain_classic.agents.output_parsers import ToolsAgentOutputParser

In [43]:
# 使用原模版创建参数
model = ChatOpenAI(
    base_url=base_url,
    model="deepseek-v4-flash",
    api_key=api_key,
    temperature=0
).bind(
    tools=[
        {
            "type":"function",
            "function": fn
        } for fn in functions
    ],
    tool_choice="auto"
)

In [44]:
# 重新创建链
chain = prompt | model | ToolsAgentOutputParser()

In [45]:
response = chain.invoke({"input":"what is the weather in sf right now?"})

In [46]:
type(response[0])

langchain_classic.agents.output_parsers.tools.ToolAgentAction

In [47]:
response[0].tool

'get_current_temperature'

In [48]:
response[0].tool_input

{'latitude': 37.7749, 'longitude': -122.4194}

In [50]:
get_current_temperature.invoke(response[0].tool_input) # 需要通过invoke调用

In [52]:
result = chain.invoke({"input": "hi"})

In [53]:
type(result) # 返回的是agent完成

langchain_core.agents.AgentFinish

In [55]:
result.return_values["output"]

"Hi there! 😊 How can I help you today? I can look up information from Wikipedia or check the current temperature for a location — or just chat. What's on your mind?"

In [49]:
# 具体显示大模型是否实现了函数调用，表示为代理完成或者代理操作

In [64]:

from langchain_classic.schema.agent import AgentFinish
def route(result):
    if isinstance(result, AgentFinish): # 定义路由函数，判断chat是否为AgentFinish
        # isinstance用于判断Result是否是AgentFinish的实例
        return result.return_values["output"]
    else :
        tools = {
            "search_wikipedia": search_wikipedia,
            "get_current_temperature": get_current_temperature,
        }
        return tools[result[0].tool].invoke(result[0].tool_input)
    # 新版本当中返回的是一个列表，tools用于获取工具，invoke调用返回值当中的工具输入

In [65]:
chain = prompt | model | ToolsAgentOutputParser() | route # 路由函数

In [66]:
result = chain.invoke({"input": "What is the weather in san francisco right now?"})

In [67]:
result

'The current temperature is 13.0 ℃'

In [69]:
result = chain.invoke({"input":"What is langchain?"})

In [71]:
result # 调用wikipedia需要开启代理

"Page:LangChain\nSummary:LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.\n\n\n\nPage:Agent harness\nSummary:An agent harness, also known as agent scaffolding, is the software infrastructure surrounding a large language model (LLM) that enables it to operate as an AI agent. It manages tool use, memory, state persistence, execution environments and feedback loops, as opposed to the model's internal reasoning.  The UK's AI Security Institute described an AI agent as being formed by the model plus the scaffolding back in 2023.  The relationship can be expressed as agent = model + harness.\nBecause an LLM is stateless, unaided, and produces only text, the harness is what allows a model to take actions over multiple steps,

In [72]:
chain.invoke({"input":"hi!"})

"Hi there! 👋 How can I help you today?\n\nI can assist with essays and writing — brainstorming, outlining, drafting, or polishing arguments. I've also got tools for looking things up on Wikipedia and checking current temperatures if you need them.\n\nWhat are you working on?"